In [1]:
import torch
import time
import keyboard
from IPython.display import clear_output

from camera_wrapper import CameraWrapper
from hand_tracker import HandTracker
from onlineKmeans import OnlineKmeans
from live_dataset import LiveDataset
from autoencoder import HumanSignalAutoencoder
from ae_trainer import AETrainer
from hand_visualizer import HandVisualizer

In [2]:
# settings

In [3]:
# architecture settings
human_signal_shape = (1, 21, 3) # 1 hand * 21 tracking points * 3 tracking dims
autoencoder_bottleneck = 32

# region classifier settings
okm_num_centroids = 2

# live dataset settings
queue_size_per_class = 50

# autoenc training settings
lr = 0.0001
ae_batch_size = 32
ae_steps_per_frame = 10

# torch settings
run_device = torch.device("cuda")

In [4]:
# ── preprocessing: translation-invariant bone offsets ─────────────────────────
# converts absolute mediapipe coords to parent-relative displacements.
# wrist becomes (0,0,0); each joint is its offset from its parent.
# this way the AE encodes hand *shape*, not position in frame.

_TOPOLOGY = [
    (0, 1), (1, 2), (2, 3), (3, 4),       # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),       # index
    (0, 9), (9, 10), (10, 11), (11, 12),   # middle
    (0, 13), (13, 14), (14, 15), (15, 16), # ring
    (0, 17), (17, 18), (18, 19), (19, 20), # pinky
]

def to_bone_offsets(tensor):
    out = tensor.clone()
    for h in range(out.shape[0]):
        out[h, 0] = 0.0
        for parent, child in _TOPOLOGY:
            out[h, child] = tensor[h, child] - tensor[h, parent]
    return out

In [5]:
camera = CameraWrapper(camera_index=1)
hand_tracker = HandTracker("./hand_landmarker.task")

In [6]:
true_visualizer = HandVisualizer('True Hand')
decoded_visualizer = HandVisualizer('Decoded Hand')

In [7]:
autoenc = HumanSignalAutoencoder(human_signal_shape, autoencoder_bottleneck).to(run_device)
region_classifier = OnlineKmeans(okm_num_centroids, run_device)
live_dataset = LiveDataset(okm_num_centroids, queue_size_per_class)
ae_trainer = AETrainer(autoenc, lr=lr, device=run_device)

In [ ]:
# ── main loop ─────────────────────────────────────────────────────────────────
start_time = time.time()
round_robin_counter = 0
loss_val = float('nan')

while not keyboard.is_pressed('q'):
    # get a pic from the camera
    img = camera()

    # run the hand tracker on the camera image
    human_signal = hand_tracker(img, int((time.time() - start_time) * 1000)).to(run_device)

    # check that we got not more or less than 1 hand in the result
    if human_signal.shape != torch.Size(human_signal_shape):
        continue

    # preprocess: strip absolute position, keep only hand shape
    human_signal = to_bone_offsets(human_signal)

    # ── encode current frame (no grad, for classification + storage) ──────────
    with torch.no_grad():
        latent = autoenc.encode(human_signal.unsqueeze(0)).squeeze(0)

    # ── classify via OKM ──────────────────────────────────────────────────────
    class_idx, dists = region_classifier.classify(latent)

    # before OKM is initialized, distribute round-robin across all classes
    if class_idx is None:
        class_idx = round_robin_counter % okm_num_centroids
        round_robin_counter += 1

    # store in dataset
    live_dataset.apply_datapoint(latent, human_signal, class_idx)

    # ── AE training steps (balanced sampling across classes) ──────────────────
    for _ in range(ae_steps_per_frame):
        batch = live_dataset.get_random_human_sigs(ae_batch_size)
        if batch is not None:
            loss_val = ae_trainer.train_step(batch)

    # ── OKM training (full dataset) ──────────────────────────────────────────
    all_latents = live_dataset.get_random_latents(live_dataset.total_size())
    region_classifier.retrain(all_latents)

    # ── update visuals ────────────────────────────────────────────────────────
    true_visualizer.update(human_signal.detach().cpu())
    with torch.no_grad():
        decoded = autoenc(human_signal.unsqueeze(0)).squeeze(0)
    decoded_visualizer.update(decoded.detach().cpu())

    clear_output(wait=True)
    print(f"Class: {class_idx} | Loss: {loss_val:.6f} | Queues: {live_dataset.get_queue_sizes()}")

Class: 0 | Loss: 0.000055 | Queues: [50, 50]
